# lijlk㕸加盟

## 是虽另

In [1]:
a=2
b=5
print(a*b)

10


In [1]:
a=2
b=5
print(a*b)

a=2
b=5

print(a*b)

10


In [1]:
import socket
import threading

def receive_messages(client_socket):
    while True:
        try:
            message = client_socket.recv(1024).decode('utf-8')
            print(message)
        except:
            break

client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.connect(('localhost', 5555))

receive_thread = threading.Thread(target=receive_messages, args=(client,))
receive_thread.start()

while True:
    message = input()
    client.send(message.encode('utf-8'))

干枯
槈


KeyboardInterrupt: Interrupted by user

In [4]:
import socket
import threading
import json
import os
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QHBoxLayout, QTextEdit, QLineEdit, QPushButton, QLabel, QFileDialog
from PyQt5.QtCore import Qt, pyqtSignal, QObject

class SignalHandler(QObject):
    update_chat = pyqtSignal(str)
    update_users = pyqtSignal(list)

class ChatClient(QWidget):
    def __init__(self):
        super().__init__()
        self.client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.signal_handler = SignalHandler()
        self.initUI()
        
    def initUI(self):
        self.setWindowTitle('聊天客户端')
        self.setGeometry(300, 300, 500, 500)
        
        layout = QVBoxLayout()
        
        self.chat_area = QTextEdit()
        self.chat_area.setReadOnly(True)
        layout.addWidget(self.chat_area)
        
        input_layout = QHBoxLayout()
        self.input_box = QLineEdit()
        self.send_button = QPushButton('发送')
        self.send_button.clicked.connect(self.send_message)
        input_layout.addWidget(self.input_box)
        input_layout.addWidget(self.send_button)
        layout.addLayout(input_layout)
        
        file_layout = QHBoxLayout()
        self.file_button = QPushButton('发送文件')
        self.file_button.clicked.connect(self.send_file)
        file_layout.addWidget(self.file_button)
        layout.addLayout(file_layout)
        
        self.setLayout(layout)
        
        self.signal_handler.update_chat.connect(self.update_chat)
        
    def connect_to_server(self):
        self.client.connect(('localhost', 5555))
        receive_thread = threading.Thread(target=self.receive_messages)
        receive_thread.start()
        
    def receive_messages(self):
        while True:
            try:
                data = self.client.recv(1024).decode('utf-8')
                message = json.loads(data)
                if message['type'] == 'message' or message['type'] == 'private_message':
                    self.signal_handler.update_chat.emit(f"{message['sender']}: {message['content']}")
                elif message['type'] == 'file_notice':
                    self.signal_handler.update_chat.emit(f"{message['sender']} 发送了文件: {message['filename']}")
            except:
                break
                
    def send_message(self):
        message = self.input_box.text()
        self.client.send(json.dumps({'type': 'message', 'to': 'all', 'content': message}).encode('utf-8'))
        self.input_box.clear()
        
    def send_file(self):
        file_path, _ = QFileDialog.getOpenFileName(self, "选择文件")
        if file_path:
            filename = os.path.basename(file_path)
            filesize = os.path.getsize(file_path)
            self.client.send(json.dumps({'type': 'file', 'to': 'all', 'filename': filename, 'filesize': filesize}).encode('utf-8'))
            with open(file_path, 'rb') as f:
                self.client.sendall(f.read())
                
    def update_chat(self, message):
        self.chat_area.append(message)
        
    def login(self, username, password):
        self.client.send(json.dumps({'type': 'login', 'username': username, 'password': password}).encode('utf-8'))

if __name__ == '__main__':
    app = QApplication([])
    client = ChatClient()
    client.show()
    client.connect_to_server()
    client.login('test_user', 'test_password')  # 这里应该有一个登录界面
    app.exec_() 